# From control flow to pandas: a course-catalog question

**Master's in Business Data Science · M1 · Wednesday 9 September 2026**

This is the first 80-minute core notebook in a two-notebook sequence. We begin with a small list of course dictionaries, then ask the same kind of question with NumPy and pandas:

> Which courses are both in Web Development and popular enough to investigate?

The second notebook adds 110 minutes of analysis. Together with breaks and wrap-up, the block is 225 minutes.

## Learning goals

By the end of this notebook, you can:

- use a `for` loop, `if`/`elif`/`else`, and a function to process records;
- describe a NumPy array using `shape` and `dtype`, then calculate with a whole array and build a boolean mask;
- inspect a pandas `DataFrame` and distinguish column labels (`.loc`) from integer positions (`.iloc`);
- combine conditions with parentheses and `&`, select rows, assign with `.loc`, and summarize with `groupby`.

Run the notebook from top to bottom. The small sample gives predictable checkpoints; the historical catalog gives practice with a real file.

### Teaching plan (80-minute core)

| Block | Focus | Minutes |
| --- | --- | ---: |
| 1 | Python loop, conditionals, function | 20 |
| 2 | NumPy arrays, shape, dtype, arithmetic, masks | 20 |
| 3 | pandas inspection, labels vs positions, masks, assignment | 25 |
| 4 | `groupby`, business question, check-out exercises | 15 |

The timings include the short predict/run/modify/check tasks. If you need to pause, leave the optional extension for later.


In [ ]:
# Imports and versions: the notebook uses only common Colab/runtime packages.
import sys
import numpy as np
import pandas as pd

print(f"Python {sys.version.split()[0]}")
print(f"NumPy {np.__version__}")
print(f"pandas {pd.__version__}")


## 1. Python gives us explicit control (20 minutes)

Before a data library, a record can simply be a dictionary. A list of dictionaries makes the work visible: loop over each course, inspect fields, and decide what to do.

![A visual bridge from a loop to a boolean mask](https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/media/M1_2026/loop_to_mask.png)

The same idea can be drawn in words: a loop asks about one record at a time; a boolean mask asks the same question for every row at once. We will build that bridge carefully. For a language reference, see the [Python control flow tutorial](https://docs.python.org/3/tutorial/controlflow.html).


The five course titles, subjects, prices, and enrollment numbers below are **synthetic illustrative values** created for teaching. They are not Udemy metrics. The historical Udemy catalog appears later, with its own source note.


In [ ]:
sample_courses = [
    {"course_title": "Python Basics Lab", "subject": "Web Development", "price": 0, "num_subscribers": 1500},
    {"course_title": "Marketing Analytics Intro", "subject": "Business Finance", "price": 200, "num_subscribers": 1200},
    {"course_title": "Web Projects Studio", "subject": "Web Development", "price": 200, "num_subscribers": 1100},
    {"course_title": "Security Lab", "subject": "Web Development", "price": 0, "num_subscribers": 950},
    {"course_title": "Finance Spreadsheet Skills", "subject": "Business Finance", "price": 150, "num_subscribers": 500},
]

print(f"Records: {len(sample_courses)}")
print(sample_courses[0])


### 1.1 Loop and `if`: make the decision visible

**Predict:** what total will the loop print? Run the cell, then modify the condition in a copy (for example, select only courses with at least 1,000 subscribers) to see how the answer changes.


In [ ]:
web_subscribers = 0
for course in sample_courses:
    if course["subject"] == "Web Development":
        web_subscribers += course["num_subscribers"]

print("Web Development subscribers:", web_subscribers)


### 1.2 `if`/`elif`/`else`: turn a number into a business label

The rules below are deliberately simple. A function will make the rules reusable.


In [ ]:
for course in sample_courses:
    price = course["price"]
    if price == 0:
        tier = "Free"
    elif price <= 100:
        tier = "Standard"
    else:
        tier = "Premium"
    print(f"{course['course_title']}: {tier}")


In [ ]:
def price_tier(price):
    '''Return a readable tier for a non-negative course price.'''
    if price == 0:
        return "Free"
    if price <= 100:
        return "Standard"
    return "Premium"

for course in sample_courses:
    print(course["course_title"], "->", price_tier(course["price"]))


A note on business language: `price * num_subscribers` can be a useful teaching proxy for a price-times-enrollment calculation. It is **not actual revenue**: we do not know discounts, refunds, subscriptions, platform fees, taxes, or whether enrollments are unique people. We will label it as a proxy whenever it appears.


> **Judgement call.** Naming this `price_times_enrollments_proxy` rather than `revenue` is the whole exercise. Nobody observed a transaction here. The moment it gets called revenue in a slide, it starts being treated as money, and no amount of correct arithmetic downstream fixes that. What you call a column is a claim about what it measures.


In [ ]:
for course in sample_courses:
    price_times_enrollments_proxy = course["price"] * course["num_subscribers"]
    print(f"{course['course_title']}: {price_times_enrollments_proxy:,} price-times-enrollments (proxy)")


### Practice 1 — predict, run, modify, check

Write a loop that collects the titles of free courses into `free_titles`. The starter cell is intentionally unfinished but safe to run. After trying it, compare with the solution key at the bottom.


In [ ]:
# TODO: replace None with your loop result.
free_titles = None
print("Your free-course titles:", free_titles)


**Checkpoint:** the loop should print 3,550 for the untouched sample. The completed free-title result should contain Python Basics Lab and Security Lab, in that order. Check both the condition and the field you append.


## 2. NumPy makes numeric work array-shaped (20 minutes)

A NumPy `ndarray` stores values of one data type in a regular shape. That regularity lets us calculate across all values at once. See the [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html) for `shape`, `dtype`, and array operations.


In [ ]:
# [course["price"] for course in sample_courses] is the loop from section 1 on one line:
# it walks the records and collects one value from each.
prices = np.array([course["price"] for course in sample_courses], dtype=float)
enrollments = np.array([course["num_subscribers"] for course in sample_courses], dtype=int)

print("prices:", prices)
print("prices shape and dtype:", prices.shape, prices.dtype)
print("enrollments shape and dtype:", enrollments.shape, enrollments.dtype)


In [ ]:
python_numbers = [1, 2, 3]
numpy_numbers = np.array(python_numbers)
print("Python list * 2:", python_numbers * 2)
print("NumPy array * 2:", numpy_numbers * 2)


A Python list repeats its contents when multiplied by 2. A NumPy array multiplies each value by 2. This element-wise behavior is the small step from explicit loops to array arithmetic.


`shape == (5,)` means five values along one axis. `dtype` describes how NumPy stores each value. Arithmetic is element by element, so the result below lines up with the five courses.


In [ ]:
price_times_enrollments_proxy = prices * enrollments
print("Proxy values:", price_times_enrollments_proxy)
print("Total proxy:", price_times_enrollments_proxy.sum())


### A boolean mask is a vector of decisions

**Predict:** which positions will be `True` for `enrollments >= 1000`? Then run the cell. Indexing with the mask keeps only matching values.


In [ ]:
popular_mask = enrollments >= 1000
print("Mask:", popular_mask)
print("Popular enrollments:", enrollments[popular_mask])


### Practice 2 — combine two array conditions

Use `&` to find Web Development courses that are popular. NumPy has no course subject array yet, so create one from the sample first. Parentheses around each comparison keep the order clear.


In [ ]:
subjects = np.array([course["subject"] for course in sample_courses])

# TODO: replace None with one boolean array.
web_and_popular = None
print("Your mask:", web_and_popular)


**Checkpoint:** the completed mask should be `[ True False  True False False]` for the untouched sample. If you see an error, check that each comparison is in parentheses before `&`.


### Bridge: the same records as a DataFrame

The list of dictionaries already has rows and fields. `pd.DataFrame` turns those records into labeled columns, which is the shape pandas expects. We use the name `df` here, then replace it with the historical catalog in the next section.


In [ ]:
df = pd.DataFrame(sample_courses)
display(df)
print("Columns:", df.columns.tolist())


## 3. The same question in pandas (25 minutes)

`pandas.DataFrame` adds labels and column names to a rectangular table. The catalog is a historical teaching dataset from the [aaubs/ds-master repository](https://github.com/aaubs/ds-master). It describes a past Udemy course catalog, so it is not a 2026 market snapshot. `num_subscribers` is an enrollment count in the source; it is not necessarily unique people or actual sales.


In [ ]:
# Step 1 — list the places the catalog file may live, and look for it.
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/udemy_courses_info.csv"
LOCAL_DATA_CANDIDATES = [
    Path("../data/udemy_courses_info.csv"),
    Path("../ds-master/data/udemy_courses_info.csv"),
    Path("data/udemy_courses_info.csv"),
    Path("ds-master/data/udemy_courses_info.csv"),
]

local_path = None
for path in LOCAL_DATA_CANDIDATES:
    if path.exists():
        local_path = path
        break

print("Local file found:", local_path)


In [ ]:
# Step 2 — read the catalog from the local file if we have one, otherwise from the URL.
if local_path is not None:
    df = pd.read_csv(local_path)
    data_source = str(local_path)
else:
    try:
        df = pd.read_csv(DATA_URL)
        data_source = DATA_URL
    except Exception as exc:
        raise FileNotFoundError(
            "Could not load the catalog from the URL, and no local data/udemy_courses_info.csv was found."
        ) from exc

print("Data source:", data_source)
print("Shape:", df.shape)


The source choice is explicit: the cell reports whether it used the repository URL or a local file. It does not silently substitute another dataset.


In [ ]:
# Step 1 — what are the columns, and how big is the table?
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


In [ ]:
# Step 2 — look at a few records.
display(df.head(3))


In [ ]:
# Step 3 — check the type and the non-missing count of every column.
df.info()


In [ ]:
# Numeric summaries are data-dependent; read them as a profile, not a fixed answer.
display(df[["price", "num_subscribers", "num_reviews"]].describe())


### 3.1 Labels versus positions: `.loc` and `.iloc`

`.loc` selects by row/column labels; `.iloc` selects by integer positions. The [pandas indexing guide](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html) documents both. This tiny table makes the distinction predictable.


In [ ]:
demo = pd.DataFrame(
    {"course": ["A", "B", "C"], "subscribers": [10, 20, 30]},
    index=["course_A", "course_B", "course_C"],
)
display(demo)
print(".loc label:", demo.loc["course_B", "subscribers"])
print(".iloc position:", demo.iloc[1, 1])


Both values are 20 here because `course_B` happens to be in position 1. The reason to learn the difference is that labels can be reordered or changed while positions always mean row 0, row 1, and so on.


### 3.2 Boolean masks and assignment with `.loc`

A pandas comparison creates a boolean Series aligned to the DataFrame's row labels. For multiple conditions, put each comparison in parentheses and join them with `&`.


In [ ]:
# Step 1 — build one mask per condition, and look at each one on its own.
is_web = df["subject"] == "Web Development"
is_popular = df["num_subscribers"] >= 100000
print("Web Development rows:", int(is_web.sum()))
print("Rows with 100,000+ subscribers:", int(is_popular.sum()))


In [ ]:
# Step 2 — combine the two masks with & (both conditions must hold).
web_and_popular_df = is_web & is_popular
print("Rows matching both conditions:", int(web_and_popular_df.sum()))


In [ ]:
# Step 3 — use the mask to select rows, and a column list to select columns.
display(df.loc[web_and_popular_df, ["course_title", "subject", "num_subscribers"]].head(5))


In [ ]:
# Assignment with .loc changes only matching rows.
# (For a plain True/False flag, df["investigate"] = web_and_popular_df is enough.
#  The .loc form is what you need when you set a value on some rows only.)
df["investigate"] = False
df.loc[web_and_popular_df, "investigate"] = True
print(df["investigate"].value_counts())
display(df.loc[df["investigate"], ["course_title", "num_subscribers", "investigate"]].head(5))


### Practice 3 — make a paid, popular mask

Find courses that are paid (`is_paid` is `True`) and have at least 100,000 subscribers. Then use `.loc` to display only `course_title`, `price`, and `num_subscribers`. The result count depends on the historical file; inspect it rather than memorizing a number.


In [ ]:
# TODO: replace None with a boolean Series, then use it in the display line.
paid_and_popular = None
print("Rows selected:", paid_and_popular)
if paid_and_popular is not None:
    display(df.loc[paid_and_popular, ["course_title", "price", "num_subscribers"]].head())


**Hint:** build one comparison for payment status and one for subscriber count. Combine them with `&` and put each comparison in parentheses. The solution key is at the end.


## 4. Split, apply, combine with `groupby` (15 minutes)

`groupby` expresses a common business question: split rows into groups, apply a summary to each group, then combine the results. The visual below shows that flow.

![Split, apply, combine with groupby](https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/media/M1_2026/split_apply_combine.png)

In words: split courses by `subject`; apply `sum` or `mean` to `num_subscribers`; combine the summaries into a labeled Series or DataFrame. See the [pandas groupby documentation](https://pandas.pydata.org/docs/reference/groupby.html).


In [ ]:
subscribers_by_subject = (
    df.groupby("subject")["num_subscribers"]
      .sum()
      .sort_values(ascending=False)
)
display(subscribers_by_subject)


A total can be useful, but a large subject may simply have more courses. Compare it with the mean and the course count.


> **Judgement call.** Totals and averages rank these subjects differently, and neither ranking is wrong. The question is which one answers what somebody actually asked. A tool will compute whichever you name; deciding which one belongs in the answer is not something you can hand off, because it depends on a question that only exists in your head.


In [ ]:
subject_summary = (
    df.groupby("subject")
      .agg(
          courses=("course_id", "count"),
          total_subscribers=("num_subscribers", "sum"),
          average_subscribers=("num_subscribers", "mean"),
      )
      .sort_values("average_subscribers", ascending=False)
)
display(subject_summary)


### Practice 4 — answer one constrained question

Which subject has the highest average number of subscribers among `Beginner Level` courses? Complete the starter cell with a filter, a `groupby`, and a mean. Print the whole labeled result before selecting the first row.


In [ ]:
# TODO: complete each line. These placeholders run safely before you fill them in.
beginner = None
average_by_subject = None
print("Beginner rows:", beginner)
print("Average subscribers by subject:", average_by_subject)


## Check-out

You should now be able to explain the same pattern at three levels:

1. Python: loop over records and make an `if` decision.
2. NumPy: calculate on an array and keep values selected by a boolean mask.
3. pandas: use labeled columns, combine masks, assign with `.loc`, and summarize with `groupby`.

For the next notebook, bring one question about this catalog that would benefit from a second grouping variable or a chart. The dataset remains historical context, not evidence about the 2026 market.


## Hints (after attempting the practices)

- **Practice 1:** start with `free_titles = []`; loop through `sample_courses`; append `course["course_title"]` when `course["price"] == 0`.
- **Practice 2:** combine `(subjects == "Web Development")` with `(enrollments >= 1000)` using `&`.
- **Practice 3:** `df["is_paid"]` is already a True/False column, so it can be used as a mask directly; compare `df["num_subscribers"]` to `100000` and join the two Series with `&`.
- **Practice 4:** filter with `df["level"] == "Beginner Level"`, then group by `subject` and take the mean of `num_subscribers`.


## Solution key

Run these only after your attempt. Each solution is separate so you can compare one task at a time.


In [ ]:
free_titles = []
for course in sample_courses:
    if course["price"] == 0:
        free_titles.append(course["course_title"])
print(free_titles)


In [ ]:
web_and_popular = (subjects == "Web Development") & (enrollments >= 1000)
print(web_and_popular)


In [ ]:
# is_paid already holds True/False, so it goes straight into the &.
paid_and_popular = df["is_paid"] & (df["num_subscribers"] >= 100000)
display(df.loc[paid_and_popular, ["course_title", "price", "num_subscribers"]].head())


In [ ]:
beginner = df.loc[df["level"] == "Beginner Level"]
average_by_subject = (
    beginner.groupby("subject")["num_subscribers"]
            .mean()
            .sort_values(ascending=False)
)
print(average_by_subject)
print("Top subject:", average_by_subject.index[0])


### Optional extension (only if time remains)

Use vectorized selection to create a price label for the whole catalog. This extension is a chance to connect the function rules to array-style logic; it is not required for the 80-minute core.


In [ ]:
# pd.cut turns a number into a named band. The bins are the cut points and
# each band is closed on the right: (-1, 0] is Free, (0, 100] is Standard, the rest Premium.
# This is the price_tier function from section 1.2, written as one pandas call.
df["price_tier"] = pd.cut(
    df["price"],
    bins=[-1, 0, 100, float("inf")],
    labels=["Free", "Standard", "Premium"],
)
display(df[["course_title", "price", "price_tier"]].head())
